# Migration: Dept_pkg ETL Session
**Source:** ODI TARGET SQL
**Target:** Databricks Spark SQL Jupyter Notebook
**Conversion Date:** 2023-10-27

In [ ]:
dbutils.widgets.text("v_lst_dt", "")
dbutils.widgets.text("ODI_SESS_NO", "f65dfa48-bc1b-4502-9e42-d72c902cd1cf")
dbutils.widgets.text("ETL_PROC_WID", "")

## Parameter Handling
SCEN_TASK_NO {1, 2}

In [ ]:
%sql
-- Fetch last run date from log table
CREATE OR REPLACE TEMPORARY VIEW v_last_run_date AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS last_dt;

-- Update log table with current run date
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

## Staging Table Setup (C$)
SCEN_TASK_NO {30, 40}

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

CREATE TABLE workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP
)
USING DELTA;

## Data Extraction to Staging
SCEN_TASK_NO {50, 60}

In [ ]:
%sql
INSERT INTO workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    SRC_DEPARTMENTS.DEPARTMENT_ID   AS DEPARTMENT_ID,
    SRC_DEPARTMENTS.DEPARTMENT_NAME AS DEPARTMENT_NAME,
    SRC_DEPARTMENTS.MANAGER_ID      AS MANAGER_ID,
    SRC_DEPARTMENTS.LOCATION_ID     AS LOCATION_ID,
    SRC_DEPARTMENTS.LAST_UPD_DT     AS LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS
WHERE (1=1)
  AND (SRC_DEPARTMENTS.LAST_UPD_DT >= to_date('${v_lst_dt}', 'dd-MM-yy'));

OPTIMIZE workspace.odi_trg.c_0filter;

## Flow Table Setup (I$)
SCEN_TASK_NO {80, 90}

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    IND_UPDATE      STRING,
    ODI_ROW_ID      STRING
)
USING DELTA;

## Populate Flow Table
SCEN_TASK_NO {100, 110, 120}

In [ ]:
%sql
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE,
    ODI_ROW_ID
)
SELECT 
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE,
    CAST(monotonically_increasing_id() AS STRING) AS ODI_ROW_ID
FROM (
    SELECT 
        FILTER_A.DEPARTMENT_ID AS DEPARTMENT_ID,
        FILTER_A.DEPARTMENT_NAME AS DEPARTMENT_NAME,
        FILTER_A.MANAGER_ID AS MANAGER_ID,
        FILTER_A.LOCATION_ID AS LOCATION_ID,
        FILTER_A.LAST_UPD_DT AS LAST_UPD_DT,
        'I' AS IND_UPDATE
    FROM workspace.odi_trg.c_0filter FILTER_A
    WHERE (1=1)
) S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_departments T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
      AND (T.DEPARTMENT_NAME = S.DEPARTMENT_NAME OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL))
      AND (T.MANAGER_ID = S.MANAGER_ID OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))
      AND (T.LOCATION_ID = S.LOCATION_ID OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL))
      AND (T.LAST_UPD_DT = S.LAST_UPD_DT OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
);

SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

## Check and Error Tables Setup
SCEN_TASK_NO {130, 140, 150, 160}

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'ODI_TRG'
  AND ORIGIN = '(7)test.Dept_MAp'
  AND ERR_TYPE = 'F';

CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING,
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP,
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'S' AND 'F' = 'S')
   OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

## Constraint Validation (PK and Not Null)
SCEN_TASK_NO {180, 190}

In [ ]:
%sql
-- PK Violation Detection
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}',
    TRG_DEPARTMENTS.ODI_ROW_ID,
    'F',
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',
    TRG_DEPARTMENTS.DEPARTMENT_ID,
    TRG_DEPARTMENTS.DEPARTMENT_NAME,
    TRG_DEPARTMENTS.MANAGER_ID,
    TRG_DEPARTMENTS.LOCATION_ID,
    TRG_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow TRG_DEPARTMENTS
WHERE EXISTS (
    SELECT 1
    FROM workspace.odi_trg.i_trg_departments_flow SUB
    WHERE SUB.DEPARTMENT_ID = TRG_DEPARTMENTS.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

-- Not Null Violation Detection
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}',
    ODI_ROW_ID,
    'F',
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

## Cleanse Flow Table and Log Errors
SCEN_TASK_NO {210, 220}

In [ ]:
%sql
-- Remove erroneous records from Flow table using MERGE DELETE pattern
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING (
    SELECT ODI_ROW_ID
    FROM workspace.odi_trg.e_trg_departments
    WHERE ODI_SESS_NO = '${ODI_SESS_NO}'
) AS E
ON T.ODI_ROW_ID = E.ODI_ROW_ID
WHEN MATCHED THEN DELETE;

-- Insert summary into audit table
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT
    'ODI_TRG',
    'TRG_DEPARTMENTS',
    'workspace.odi_trg.trg_departments',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    COUNT(1)
FROM workspace.odi_trg.e_trg_departments E
WHERE E.ODI_ERR_TYPE = 'F'
  AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY 
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE;

## Integration (Flagging and MERGE)
SCEN_TASK_NO {230, 270, 280}

In [ ]:
%sql
-- Flag records for update using MERGE (replacing tuple IN)
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

-- Final MERGE into target table
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID      = S.MANAGER_ID,
    T.LOCATION_ID     = S.LOCATION_ID,
    T.LAST_UPD_DT     = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
) VALUES (
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

## Final Cleanup
SCEN_TASK_NO {340, 360}

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;